In [15]:
import sys
sys.path.append('../')

from utils_states import (
    compress_state,
    decompress_state
)
from utils_m1_seniority import (
    seniority_solving_clifford_operator,
    group_odds_and_evens,
    taper_hamiltonian,
    project_out_seniority_symmetries
)
from utils_m2_factorize import (
    expand_tensor_product_for_incomplete_qubit_set
)
from utils_basic import (
    compute_product_of_unitaries,
    random_pauli_hamiltonian,
    apply_unitary_product
)
import numpy as np
from numpy.random import uniform
from openfermion import (
    QubitOperator,
    get_sparse_operator
)

def random_fixed_seniority_state(config, Norb):

    assert len(config) == Norb

    factors   = []
    factors_t = []

    for i in range(Norb):

        state   = np.zeros(4)
        state_t = np.zeros(2)

        if config[i] == 0:
            val                  = uniform(-1, 1)
            state[int('00', 2)]  = val
            state_t[int('0', 2)] = val

            val                  = uniform(-1, 1)
            state[int('11', 2)]  = val
            state_t[int('1', 2)] = val

        elif config[i] == 1:
            val                  = uniform(-1, 1)
            state[int('10', 2)]  = val
            state_t[int('0', 2)] = val

            val                  = uniform(-1, 1)
            state[int('01', 2)]  = val
            state_t[int('1', 2)] = val

        factors.append(state)
        factors_t.append(state_t)

    psi = factors[0]
    for factor in factors[1:]:
        psi = np.kron(psi, factor)

    psi_t = factors_t[0]
    for factor_t in factors_t[1:]:
        psi_t = np.kron(psi_t, factor_t)

    config_state                                            = np.zeros(2 ** Norb)
    config_state[int(''.join([str(z) for z in config]), 2)] = 1

    return psi, psi_t, config_state


Test `compress_state` and `decompress_state`

Test: if I prepare a state with fixed seniorities, compress state should return the correct tapered state, and decompress state should return the original state.

In [33]:
for _ in range(10):

    Nqubits     = 16
    Norb        = Nqubits // 2
    config      = [1,0,1,1,0,0,1,0]
    fake_config = [0,0,1,0,0,1,0,0] # used only for verification step at the very end 

    assert len(config) == Norb
    assert config != fake_config

    factors   = []
    factors_t = []

    for i in range(Norb):

        if config[i] == 0:
            state                = np.zeros(4)
            state_t              = np.zeros(2)

            val                  = uniform(-1, 1)
            state[int('00', 2)]  = val
            state_t[int('0', 2)] = val

            val                  = uniform(-1, 1)
            state[int('11', 2)]  = val
            state_t[int('1', 2)] = val
            

        elif config[i] == 1:
            state                = np.zeros(4)
            state_t              = np.zeros(2)

            val                  = uniform(-1, 1)
            state[int('10', 2)]  = val
            state_t[int('0', 2)] = val

            val                  = uniform(-1, 1)
            state[int('01', 2)]  = val
            state_t[int('1', 2)] = val

        factors.append(state)
        factors_t.append(state_t)

    psi = factors[0]
    for factor in factors[1:]:
        psi = np.kron(psi, factor)

    psi_t = factors_t[0]
    for factor_t in factors_t[1:]:
        psi_t = np.kron(psi_t, factor_t)

    assert np.allclose(compress_state(psi), psi_t)
    assert np.allclose(psi, decompress_state(psi_t, config))
    assert not np.allclose(psi, decompress_state(psi_t, fake_config))

In [6]:
for _ in range(10):

    Nqubits     = 16
    Norb        = Nqubits // 2
    config      = [1,0,1,1,0,0,1,0]
    fake_config = [0,0,1,0,0,1,0,0] # used only for verification step at the very end 

    assert len(config) == Norb
    assert config != fake_config

    psi, psi_t, config_state = random_fixed_seniority_state(config, Norb)

    assert np.allclose(compress_state(psi), psi_t)
    assert np.allclose(psi, decompress_state(psi_t, config))
    assert not np.allclose(psi, decompress_state(psi_t, fake_config))

Test `seniority_solving_clifford_operator`:

The test is as follows. I will create a fixed seniority state $|\psi\rangle$, along with its seniority encoding computational basis state $|\vec{v}\rangle$ and tapered representation $|\psi_t\rangle$. Using the `factorization_dict` data structure, I will verify that $U_c|\psi\rangle = |\vec{v}\rangle_\text{odd} \otimes |\psi_t\rangle_\text{even}$.

In [88]:
Nqubits = 12
Norb = Nqubits // 2
config = [0,1,1,0,0,0]
assert len(config) == Norb

factors   = []
factors_t = []

for i in range(Norb):
    state = np.zeros(4)
    state_t = np.zeros(2)

    if config[i] == 0:
        val                  = uniform(-1, 1)
        state[int('00', 2)]  = val
        state_t[int('0', 2)] = val

        val                  = uniform(-1, 1)
        state[int('11', 2)]  = val
        state_t[int('1', 2)] = val

    elif config[i] == 1:
        val                  = uniform(-1, 1)
        state[int('10', 2)]  = val
        state_t[int('0', 2)] = val

        val                  = uniform(-1, 1)
        state[int('01', 2)]  = val
        state_t[int('1', 2)] = val

    factors.append(state)
    factors_t.append(state_t)

psi = factors[0]
for factor in factors[1:]:
    psi = np.kron(psi, factor)

psi_t = factors_t[0]
for factor_t in factors_t[1:]:
    psi_t = np.kron(psi_t, factor_t)
    
# transform psi using seniority solving Clifford transformation
U               = compute_product_of_unitaries(seniority_solving_clifford_operator(Nqubits))
Usparse         = get_sparse_operator(U, Nqubits) 
psi_transformed = Usparse @ psi

# recover psi_transformed from tapered psi and seniority config, using factorization_dict data structure
psi_seniority                                            = np.zeros(2 ** Norb)
psi_seniority[int(''.join([str(z) for z in config]), 2)] = 1

odds  = tuple(range(1, Nqubits, 2))
evens = tuple(range(0, Nqubits, 2))

# the seniority basis state should be encoded on even-index qubits, and the tapered state on the odd-index qubits
# note that the Clifford transformation introduces a non-trivial phase, which must be included manually in the verification since it depends on the number of qubits

factorization_dict = {
    evens : psi_seniority,
    odds  : psi_t
}
psi_transformed_recov = expand_tensor_product_for_incomplete_qubit_set(factorization_dict)

assert np.allclose(psi_transformed, -1j * psi_transformed_recov)

# the verification should not work when seniority is encoded on even qubits, unless the state is seniority-zero, in which case both give the same result

factorization_dict = {
    odds  : psi_seniority,
    evens : psi_t
}
psi_transformed_recov = expand_tensor_product_for_incomplete_qubit_set(factorization_dict)

if config != [0] * Norb:
    assert not np.allclose(np.abs(psi_transformed), np.abs(psi_transformed_recov))
else:
    assert np.allclose(np.abs(psi_transformed), np.abs(psi_transformed_recov))

In [7]:
Nqubits = 12
Norb = Nqubits // 2
config = [0,1,1,0,0,0]
assert len(config) == Norb

factors   = []
factors_t = []

psi, psi_t, psi_seniority = random_fixed_seniority_state(config, Norb)

U               = compute_product_of_unitaries(seniority_solving_clifford_operator(Nqubits))
Usparse         = get_sparse_operator(U, Nqubits) 
psi_transformed = Usparse @ psi

odds  = tuple(range(1, Nqubits, 2))
evens = tuple(range(0, Nqubits, 2))

factorization_dict = {
    evens : psi_seniority,
    odds  : psi_t
}
psi_transformed_recov = expand_tensor_product_for_incomplete_qubit_set(factorization_dict)

assert np.allclose(psi_transformed, -1j * psi_transformed_recov)

factorization_dict = {
    odds  : psi_seniority,
    evens : psi_t
}
psi_transformed_recov = expand_tensor_product_for_incomplete_qubit_set(factorization_dict)

if config != [0] * Norb:
    assert not np.allclose(np.abs(psi_transformed), np.abs(psi_transformed_recov))
else:
    assert np.allclose(np.abs(psi_transformed), np.abs(psi_transformed_recov))

Test `group_odds_and_evens`


In [4]:
Nqubits = 10

H = QubitOperator()
H += uniform(-1, 1) * QubitOperator('X0 Z1 X2 Z3 X4 Z5')
H += uniform(-1, 1) * QubitOperator('X0 X4')
H += uniform(-1, 1) * QubitOperator('Z1 X4 Z7')
H += uniform(-1, 1) * QubitOperator('')
H += uniform(-1, 1) * QubitOperator('Z9')
H += uniform(-1, 1) * QubitOperator('X0 X2 X4 Z5 Z7 X8 Z9')

print(H)

print()

print(group_odds_and_evens(H, Nqubits))

-0.8794047998117853 [] +
-0.9688819191364788 [X0 Z1 X2 Z3 X4 Z5] +
-0.9890346582772007 [X0 X2 X4 Z5 Z7 X8 Z9] +
0.08280191229014733 [X0 X4] +
0.22964078293481793 [Z1 X4 Z7] +
0.9249641168395382 [Z9]

-0.8794047998117853 [] +
-0.9890346582772007 [X0 X1 X2 X4 Z7 Z8 Z9] +
-0.9688819191364788 [X0 X1 X2 Z5 Z6 Z7] +
0.08280191229014733 [X0 X2] +
0.22964078293481793 [X2 Z5 Z8] +
0.9249641168395382 [Z9]


For two states with fixed seniority, and a random Hamiltonian, compare four ways to obtain the matrix element

1. Numerically as $\langle \psi|H|\phi \rangle$
2. In the transformed basis using a tensor product of the seniority-encoding basis state and the tapered state
3. Using the `taper_hamiltonian` function
4. Using the `project_out_seniority_symmetries` function

In [32]:
for _ in range(10):
    Nqubits    = 12
    Norb       = Nqubits // 2
    Nterms     = 500
    bra_config = [0,0,1,0,0,0]
    ket_config = [1,1,0,0,1,0]

    # Clifford Transformation

    U   = seniority_solving_clifford_operator(Nqubits)
    Usp = get_sparse_operator(compute_product_of_unitaries(U), Nqubits)

    # States

    bra, bra_t, bra_v = random_fixed_seniority_state(bra_config, Norb)
    bra_rotated       = Usp.conj().T @ bra

    ket, ket_t, ket_w = random_fixed_seniority_state(ket_config, Norb)
    ket_rotated       = Usp @ ket

    # Hamiltonian

    H                   = random_pauli_hamiltonian(Nqubits, Nterms)
    H_rotated           = apply_unitary_product(H, U)
    H_rotated_reordered = group_odds_and_evens(H_rotated, Nqubits)
    H_tapered           = taper_hamiltonian(H_rotated_reordered, bra_config, ket_config, shift_to_zero=True)
    H_tapered_onestep   = project_out_seniority_symmetries(H, Nqubits, bra_config, ket_config)

    H_sp                   = get_sparse_operator(H, Nqubits)
    H_rotated_sp           = get_sparse_operator(H_rotated, Nqubits)
    H_rotated_reordered_sp = get_sparse_operator(H_rotated_reordered, Nqubits)
    H_tapered_sp           = get_sparse_operator(H_tapered_onestep, Norb)

    # 
    #    Test 1: both tapered Hamiltonians are the same
    #

    assert H_tapered - H_tapered_onestep == QubitOperator().zero()

    #
    #    Test 2: four ways to get matrix element produce same result
    #

    element1 = bra @ H_sp @ ket
    element2 = bra_rotated @ H_rotated_sp @ ket_rotated
    element3 = np.kron(bra_v, bra_t) @ H_rotated_reordered_sp @ np.kron(ket_w, ket_t)
    element4 = bra_t @ H_tapered_sp @ ket_t

    assert np.abs(element1 - element2) < 1e-12
    assert np.abs(element1 - element3) < 1e-12
    assert np.abs(element1 - element4) < 1e-12
    assert np.abs(element2 - element3) < 1e-12
    assert np.abs(element2 - element4) < 1e-12
    assert np.abs(element3 - element4) < 1e-12
